# 🌍 AI Climate Data Analyst — Walkthrough Notebook

This notebook demonstrates the complete pipeline:
1. **Data Generation** — Synthetic raw climate data with realistic noise
2. **Auto Cleaning** — Missing values, outliers, duplicates, type coercion
3. **EDA** — Exploratory Data Analysis with Pandas
4. **AI Insights** — LangChain + Claude for natural-language findings
5. **Visualisation** — Plotly interactive charts


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, '..')  # repo root

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ Libraries loaded')

---
## 1. Generate Raw Sample Data

In [ ]:
# Generate fresh raw data
import subprocess
result = subprocess.run(['python', 'data/generate_data.py'], capture_output=True, text=True, cwd='..')
print(result.stdout)

df_raw = pd.read_csv('../data/climate_raw.csv')
print(f'\nRaw shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# Check data quality BEFORE cleaning
print('=== Missing Values ===')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])
print(f'\n=== Duplicate Rows ===' )
print(f'{df_raw.duplicated().sum()} duplicates found')
print(f'\n=== Data Types ===')
print(df_raw.dtypes)

---
## 2. Auto-Cleaning Pipeline

In [ ]:
from src.data_cleaner import ClimateDataCleaner

cleaner = ClimateDataCleaner('../data/climate_raw.csv')
df_clean = cleaner.run()

print(f'\n✅ Clean shape: {df_clean.shape}')
print(f'   Missing values remaining: {df_clean.isnull().sum().sum()}')
print(f'   Duplicates remaining:     {df_clean.duplicated().sum()}')

In [ ]:
# Full cleaning report
import json
print(json.dumps(cleaner.get_report(), indent=2, default=str))

In [ ]:
# Compare before vs after
print('=== BEFORE ===' )
print(df_raw[['temperature_c','co2_ppm','pm2_5_ugm3']].describe())
print('\n=== AFTER ===')
print(df_clean[['temperature_c','co2_ppm','pm2_5_ugm3']].describe())

---
## 3. Exploratory Data Analysis

In [ ]:
# City-level summary
df_clean.groupby('city')[['temperature_c','aqi','co2_ppm','pm2_5_ugm3']].mean().round(2).sort_values('aqi', ascending=False)

In [ ]:
# Seasonal breakdown
df_clean.groupby('season')[['temperature_c','rainfall_mm','aqi','co2_ppm']].mean().round(2)

In [ ]:
# Correlation matrix
import plotly.express as px

num_cols = ['temperature_c','humidity_pct','co2_ppm','pm2_5_ugm3',
            'aqi','rainfall_mm','wind_speed_kmh','uv_index','pressure_hpa']
corr = df_clean[num_cols].corr().round(2)

fig = px.imshow(corr, text_auto=True, color_continuous_scale='RdBu',
                color_continuous_midpoint=0, title='Correlation Matrix')
fig.update_layout(width=700, height=600)
fig.show()

In [ ]:
# Temperature distribution by season
fig = px.box(df_clean, x='season', y='temperature_c', color='season',
             title='Temperature Distribution by Season',
             color_discrete_sequence=['#00C9A7','#FFC75F','#FF6B6B','#845EC2'])
fig.show()

In [ ]:
# AQI over time
monthly = df_clean.set_index('date').resample('ME')['aqi'].mean().reset_index()
fig = px.line(monthly, x='date', y='aqi',
              title='Monthly Average AQI',
              color_discrete_sequence=['#00C9A7'])
fig.add_hline(y=100, line_dash='dash', line_color='orange',
              annotation_text='Moderate threshold')
fig.show()

In [ ]:
# PM2.5 WHO threshold analysis
WHO_LIMIT = 35
above_who = (df_clean['pm2_5_ugm3'] > WHO_LIMIT).sum()
pct = above_who / len(df_clean) * 100
print(f'{above_who} readings ({pct:.1f}%) exceed WHO PM2.5 limit of {WHO_LIMIT} µg/m³')

# By city
city_exceedance = (df_clean.groupby('city')
                   .apply(lambda x: (x['pm2_5_ugm3'] > WHO_LIMIT).mean() * 100)
                   .round(1)
                   .sort_values(ascending=False))
print('\nExceedance rate by city:')
print(city_exceedance)

---
## 4. AI Insights with LangChain + Claude

In [ ]:
# Set your API key (or export ANTHROPIC_API_KEY in your shell)
# import os; os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

from src.insights_engine import InsightsEngine

engine   = InsightsEngine()
insights = engine.generate(df_clean)

from IPython.display import Markdown, display
display(Markdown(engine.format_markdown(insights)))

In [ ]:
# Raw JSON insights
print(json.dumps(insights, indent=2))

---
## 5. Interactive Visualisations

In [ ]:
from src.visualizer import ClimateVisualizer

viz  = ClimateVisualizer(df_clean)
figs = viz.get_all_figures()

In [ ]:
figs['temperature_trend'].show()

In [ ]:
figs['aqi_by_city'].show()

In [ ]:
figs['co2_vs_temp'].show()

In [ ]:
figs['seasonal_heatmap'].show()

In [ ]:
figs['pm25_distribution'].show()

In [ ]:
figs['correlation_matrix'].show()

In [ ]:
figs['wind_rainfall'].show()

In [ ]:
figs['air_quality_pie'].show()

---
## 6. Save Cleaned Data

In [ ]:
output_path = '../data/climate_clean.csv'
df_clean.to_csv(output_path, index=False)
print(f'✅ Saved: {output_path}  ({len(df_clean):,} rows × {len(df_clean.columns)} cols)')